# Safe Bayesian Optimization：はじめての利用ガイド

このNotebookでは、**安全な実験条件を探しながら目的関数を改善する** Safe BO の基本操作を、1変数の例で順番に学びます。

### 学習する流れ

1. 入力範囲と、守りたい安全条件を決める
2. 安全と確認した初期データを用意する
3. robotorchan の `SingleTaskGP` で目的関数と安全出力を学習する
4. `ask()` で次の実験条件を提案する
5. 実験結果を `tell()` に渡し、GPを更新する
6. 安全領域の不確実性を調べる探索モードを試す

**用語：** GP（ガウス過程）は、予測値だけでなく予測の不確実性も返すモデルです。Safe BOはその不確実性を利用し、危険と予測される候補を除外します。

**重要：** この例は予測に基づく安全判定です。実機の安全を保証するものではありません。

In [ ]:
import torch
from botorch.acquisition.analytic import PosteriorMean

from robotorchan.models.standard import SingleTaskGP
from robotorchan.optim import SafeBOWorkflow
from robotorchan.semantics import ContinuousConstraint, SafetySpecification

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0], [1.0]])


def objective(x):
    return -(x - 0.3).square()


def safety_output(x):
    return x - 0.7


def fit_gp(x, y):
    model = SingleTaskGP(x, y)
    model.fit()
    model.eval()
    return model


## 1. 初期データと安全条件を準備する

ここでは入力 `x` を0〜1の範囲で動かし、目的関数 `objective(x)` を大きくしたいとします。

一方、安全出力は `g(x) = x - 0.7` とし、**`g(x) <= 0`** を守る必要があります。つまり、この例では `x <= 0.7` が実際の安全領域です（学習側にはこの境界を直接教えず、観測データから予測します）。

- `initial_x`：安全と確認した初期条件（3点）
- `initial_g`：各条件で実測した安全出力
- `ContinuousConstraint(0, 0.0)`：安全出力の0番目が0以下という条件
- `beta=4.0`：不確実性をどれほど保守的に扱うかを決める係数

`SafetySpecification` は安全条件と初期観測をまとめ、`SafeBOWorkflow` は候補提案と観測記録を担当します。

In [ ]:
initial_x = torch.tensor([[0.05], [0.10], [0.15]])
initial_g = safety_output(initial_x)
spec = SafetySpecification(
    constraints=(ContinuousConstraint(0, 0.0),),
    initial_X=initial_x,
    observed_constraints=initial_g,
    beta=4.0,
)
workflow = SafeBOWorkflow(spec, bounds)
train_x = initial_x.clone()
train_y = objective(train_x)
train_g = initial_g.clone()
print("初期安全観測:", train_x.squeeze(-1).tolist())


## 2. GPを学習して次の実験条件を提案する

目的関数と安全出力は異なるので、ここでは**2つのGP**を別々に学習します。

`SingleTaskGP` は **robotorchan版**を使用します。BoTorchの`SingleTaskGP`を継承しており、robotorchanでは`model.fit()`で学習できます。内部の予測機能はBoTorchを利用しています。

`PosteriorMean` は「予測平均が高い点を好む」BoTorchの獲得関数です。Safe BOは獲得関数を使って候補を評価する前に、GPの信頼区間に基づいて安全候補を選別します。

`num_samples=256` は安全性を調べる候補数、`seed=10` は候補生成の再現性を保つための乱数シードです。安全な候補が見つからないときは例外になります。

In [ ]:
objective_model = fit_gp(train_x, train_y)
safety_model = fit_gp(train_x, train_g)
acquisition = PosteriorMean(objective_model)
candidate = workflow.ask(
    safety_model,
    mode="objective",
    acq_function=acquisition,
    num_samples=256,
    seed=10,
)
print("提案候補:", candidate.item())


## 3. 実験結果を登録する（tell）

`ask()` は「次にどこを測るか」を決めるだけです。**実験の実行と結果の取得は利用者の役割**です。

ここでは実験を関数で模擬し、`observed_y`（目的関数の観測値）と`observed_g`（安全出力の観測値）を得ます。

`workflow.tell(candidate, observed_g)` は**安全出力の実測値**を登録します。目的関数の観測値は別途`train_y`に追加し、次回のGP学習に使用します。

In [ ]:
observed_y = objective(candidate)
observed_g = safety_output(candidate)
workflow.tell(candidate, observed_g)
train_x = torch.cat([train_x, candidate], dim=0)
train_y = torch.cat([train_y, observed_y], dim=0)
train_g = torch.cat([train_g, observed_g], dim=0)
print("実測安全制約:", observed_g.item())
print("記録した観測数:", workflow.observed_X.shape[0])


## 4. GPを更新して安全領域を探索する

新しい実測値が増えたら、`train_x`と`train_g`を使って安全GPを再学習します。**`tell()`だけではGPは自動更新されません。**

`mode="expansion"` は、現在安全と予測される点の中で、安全出力の予測不確実性が高い候補を探します。目的関数を直接最大化する`objective`モードとは役割が異なります。

この探索方法はSafeOptの厳密な「安全領域を拡大できる点」の判定ではありません。

In [ ]:
updated_safety_model = fit_gp(train_x, train_g)
expansion_candidate = workflow.ask(
    updated_safety_model,
    mode="expansion",
    num_samples=256,
    seed=11,
)
print("探索候補:", expansion_candidate.item())
workflow.tell(expansion_candidate, safety_output(expansion_candidate))


## 5. 振り返りと実運用での注意

- `SingleTaskGP(...).fit()` でモデルを学習し、`ask()`で候補を提案し、実測後に`tell()`で安全出力を記録します。
- `beta` を大きくすると通常は保守的になりますが、確率的な安全保証を自動的に得られるわけではありません。
- 初期データは安全が確認された条件を使用してください。実測ノイズやモデルの誤差にも注意が必要です。
- 実機では、独立した安全機構や運転上の制限も必要に応じて併用してください。
- 連続・整数・カテゴリ変数を組み合わせた複数候補の提案は、`46_safe_bo_mixed_q_batch.ipynb`で説明します。

詳しい数式と安全保証の限界は、Safe BOの理論ドキュメントを参照してください。